In [ ]:
import pandas as pd
import numpy as np
import math
from scipy import stats as scipy_stats
from typing import Union, List, Tuple, Callable

In [ ]:
def bin_target(df: pd.DataFrame, target_col: str) -> dict:
    """
    Discretise a target column using Scott's rule and return a mapping
    of bin_id -> list of row indices.

    Scott's rule:
        bin_width  h = 3.5 * std(x) * n^(-1/3)
        n_bins       = ceil((max(x) - min(x)) / h)

    Parameters
    ----------
    df         : input DataFrame
    target_col : name of the column to discretise

    Returns
    -------
    dict  {bin_id (int, 0-based): [row_index, ...]}
    """
    x = df[target_col].dropna().values
    n = len(x)

    if n < 2:
        raise ValueError("Need at least 2 non-null values to compute bins.")

    std = np.std(x, ddof=1)          # sample standard deviation
    if std == 0:
        raise ValueError("Target column has zero variance; cannot bin.")

    h       = 3.5 * std * (n ** (-1 / 3))          # Scott's bin width
    n_bins  = max(1, math.ceil((x.max() - x.min()) / h))

    # Cut into n_bins equal-width intervals; labels are 0-based integer bin IDs
    bin_series = pd.cut(
        df[target_col],
        bins=n_bins,
        labels=range(n_bins),
        include_lowest=True
    )

    bin_dict = {}
    for bin_id, group in df.groupby(bin_series, observed=True):
        bin_dict[int(bin_id)] = list(group.index)

    return bin_dict, n_bins

In [ ]:
# Sample DataFrame
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(100),
    "feature_b": np.random.rand(100),
    "target":    np.random.normal(loc=50, scale=15, size=100)
})

bin_map, n_bins = bin_target(df, target_col="target")

print(f"Scott's rule → {n_bins} bins\n")
for bin_id, indices in bin_map.items():
    print(f"  Bin {bin_id:>2d} ({len(indices):>3d} items): {indices[:5]}{'...' if len(indices) > 5 else ''}")

In [ ]:
DIST_MAP = {
    "normal":      scipy_stats.norm,
    "gaussian":    scipy_stats.norm,
    "exponential": scipy_stats.expon,
    "beta":        scipy_stats.beta,
    "gamma":       scipy_stats.gamma,
    "uniform":     scipy_stats.uniform,
    "lognormal":   scipy_stats.lognorm,
}

def make_pdf(
    spec: Union[
        Tuple[str, dict],
        List[Tuple[str, dict, float]]
    ]
) -> Callable:
    """
    Single distribution:
        spec = ("normal", {"loc": 0, "scale": 1})

    Mixture (weights are auto-normalised):
        spec = [
            ("normal",      {"loc": 0, "scale": 1},  0.6),
            ("exponential", {"scale": 2},             0.4),
        ]
    """
    if isinstance(spec, tuple):
        name, params = spec
        dist = DIST_MAP[name.lower()]
        return lambda x: dist.pdf(x, **params)

    total_w    = sum(w for _, _, w in spec)
    components = [(DIST_MAP[n.lower()], p, w / total_w) for n, p, w in spec]

    def mixture_pdf(x):
        return sum(w * d.pdf(x, **p) for d, p, w in components)

    return mixture_pdf

In [ ]:
def least_cardinality_bin(bin_map: dict) -> Tuple[int, int]:
    """Return (bin_id, cardinality) of the bin with the fewest items."""
    bin_id = min(bin_map, key=lambda b: len(bin_map[b]))
    return bin_id, len(bin_map[bin_id])


def _bin_centers(df: pd.DataFrame, target_col: str, n_bins: int) -> np.ndarray:
    """Midpoint of each equal-width bin over the target column range."""
    lo, hi = df[target_col].min(), df[target_col].max()
    edges  = np.linspace(lo, hi, n_bins + 1)
    return (edges[:-1] + edges[1:]) / 2

In [ ]:
def max_peak_samples(
    bin_map:    dict,
    df:         pd.DataFrame,
    target_col: str,
    n_bins:     int,
    pdf:        Callable,
) -> Tuple[int, List[int]]:
    """
    Binary search over [1, n] where n = cardinality of the least-cardinality bin.
    Finds the largest k such that sampling k items from every peak bin still
    leaves all other bins able to supply their proportionally-required counts.

    Returns
    -------
    k_max     : maximum feasible samples per peak bin
    peak_bins : bin_ids at the distribution peak (multimodal -> multiple ids)
    """
    centers   = _bin_centers(df, target_col, n_bins)
    bin_ids   = sorted(bin_map.keys())
    pdf_vals  = np.array([pdf(centers[i]) for i in bin_ids])
    peak_pdf  = pdf_vals.max()
    peak_bins = [bid for bid, pv in zip(bin_ids, pdf_vals)
                 if np.isclose(pv, peak_pdf)]

    _, n = least_cardinality_bin(bin_map)   # binary search upper bound

    def feasible(k: int) -> bool:
        for bid, pv in zip(bin_ids, pdf_vals):
            if round(k * pv / peak_pdf) > len(bin_map[bid]):
                return False
        return True

    lo, hi, best = 1, n, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            best = mid
            lo   = mid + 1
        else:
            hi   = mid - 1

    return best, peak_bins

In [ ]:
def sanity_check(
    bin_map:    dict,
    df:         pd.DataFrame,
    target_col: str,
    n_bins:     int,
    pdf:        Callable,
    k:          int,
) -> pd.DataFrame:
    """
    For each bin verify available cardinality >= required sample count,
    where required = round(k * pdf(center_i) / peak_pdf).

    Returns a report DataFrame:
        bin_id | center | pdf_value | available | required | sufficient
    """
    centers  = _bin_centers(df, target_col, n_bins)
    bin_ids  = sorted(bin_map.keys())
    pdf_vals = np.array([pdf(centers[i]) for i in bin_ids])
    peak_pdf = pdf_vals.max()

    rows = []
    for bid, pv in zip(bin_ids, pdf_vals):
        required  = round(k * pv / peak_pdf)
        available = len(bin_map[bid])
        rows.append({
            "bin_id":     bid,
            "center":     round(centers[bid], 4),
            "pdf_value":  round(pv, 6),
            "available":  available,
            "required":   required,
            "sufficient": available >= required,
        })

    report = pd.DataFrame(rows)
    all_ok = report["sufficient"].all()
    status = "PASSED ✓" if all_ok else "FAILED ✗"
    print(f"Sanity check {status} — "
          f"{report['sufficient'].sum()}/{len(report)} bins sufficient.")
    return report

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(500),
    "target":    np.random.normal(loc=50, scale=15, size=500),
})
bin_map, n_bins = bin_target(df, "target")

# --- single Gaussian ---
pdf = make_pdf(("gaussian", {"loc": 50, "scale": 15}))

lc_bin, lc_count = least_cardinality_bin(bin_map)
print(f"Least-cardinality bin: {lc_bin}  (n={lc_count})")

k_max, peak_bins = max_peak_samples(bin_map, df, "target", n_bins, pdf)
print(f"Peak bins: {peak_bins}  |  Max samples per peak bin: {k_max}\n")

report = sanity_check(bin_map, df, "target", n_bins, pdf, k_max)
display(report)

# --- bimodal mixture ---
print("\n── Mixture example ──")
pdf_mix = make_pdf([
    ("normal", {"loc": 35, "scale": 8},  0.5),
    ("normal", {"loc": 65, "scale": 8},  0.5),
])
k_mix, peaks_mix = max_peak_samples(bin_map, df, "target", n_bins, pdf_mix)
print(f"Peak bins: {peaks_mix}  |  Max samples per peak bin: {k_mix}\n")
report_mix = sanity_check(bin_map, df, "target", n_bins, pdf_mix, k_mix)
display(report_mix)